## Parameters, task values & task references

Tasks need to share state. Three mechanisms — know which fits which question.

**Job parameters** — defined at the job level, visible to *every* task. The nightly job declares a `processing_date`; every notebook reads it (via `dbutils.widgets.get` inside the notebook).

**Task values** — a task **sets** a key-value pair that downstream tasks **read**:

```python
# upstream task
dbutils.jobs.taskValues.set(key="new_file_count", value=42)

# downstream task
count = dbutils.jobs.taskValues.get(taskKey="ingest_cards", key="new_file_count")
```

**Built-in references** — usable in any task parameter template:

- `{{job.id}}`, `{{job.name}}`, `{{job.run_id}}`
- `{{job.start_time.iso_date}}` — the run's start date
- `{{tasks.<task_key>.values.<key>}}` — a task value from upstream
- `{{job.parameters.<param>}}` — a job-level parameter

**The exam's angle:** how do you pass a **date** or a **row count** from one task to the next *without writing to a side table*? A date the whole job shares → a **job parameter**; a value one task computes and the next consumes → a **task value** (`taskValues.set` / `.get`). Reaching for a temp table to shuttle a single number between tasks is the wrong answer.
